# Imports

In [1]:
import torch
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer
from transformers import TrainingArguments
import evaluate
from transformers import Trainer
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from sklearn.metrics import classification_report

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("Current device:", torch.cuda.current_device())
else:
    print("Running on CPU")

CUDA available: True
GPU name: NVIDIA GeForce RTX 4080
Current device: 0


In [3]:
# Load the XML data and convert it to a DataFrame
def xml_to_rows(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    rows = []

    for sentence in root.findall("sentence"):
        sid = sentence.get("id")
        text = sentence.find("text").text

        aspect_terms = sentence.find("aspectTerms")

        if aspect_terms is not None:
            for aspect in aspect_terms.findall("aspectTerm"):
                rows.append({
                    "id": sid,
                    "sentence": text,
                    "aspect": aspect.get("term"),
                    "polarity": aspect.get("polarity")
                })

    return pd.DataFrame(rows)

In [4]:
myXML = xml_to_rows("./Restaurants_Train_v2.xml")

In [5]:
myXML.shape

(3693, 4)

In [6]:
myXML.head()

,id,sentence,aspect,polarity
0,3121,But the staff was so horrible to us.,staff,negative
1,2777,"To be completely fair, the only redeeming fact...",food,positive
2,1634,"The food is uniformly exceptional, with a very...",food,positive
3,1634,"The food is uniformly exceptional, with a very...",kitchen,positive
4,1634,"The food is uniformly exceptional, with a very...",menu,neutral


In [7]:
csv_df = pd.read_csv("./Laptop_Train_v2.csv", encoding="utf-8")

csv_df = csv_df[[
    "id",
    "Sentence",
    "Aspect Term",
    "polarity"
]]

csv_df.columns = ["id", "sentence", "aspect", "polarity"]

In [8]:
csv_df.head()

,id,sentence,aspect,polarity
0,2339,I charge it at night and skip taking the cord ...,cord,neutral
1,2339,I charge it at night and skip taking the cord ...,battery life,positive
2,1316,The tech guy then said the service center does...,service center,negative
3,1316,The tech guy then said the service center does...,"""sales"" team",negative
4,1316,The tech guy then said the service center does...,tech guy,neutral


In [9]:
csv_df.shape

(2358, 4)

# Merge the datasets

In [20]:
final_df = pd.concat([csv_df, myXML], ignore_index=True)

In [21]:
final_df.head()

,id,sentence,aspect,polarity
0,2339,I charge it at night and skip taking the cord ...,cord,neutral
1,2339,I charge it at night and skip taking the cord ...,battery life,positive
2,1316,The tech guy then said the service center does...,service center,negative
3,1316,The tech guy then said the service center does...,"""sales"" team",negative
4,1316,The tech guy then said the service center does...,tech guy,neutral


In [22]:
final_df["sentence"].iloc[1]

'I charge it at night and skip taking the cord with me because of the good battery life.'

In [23]:
final_df["sentence"].iloc[1]

'I charge it at night and skip taking the cord with me because of the good battery life.'

In [24]:
final_df.shape

(6051, 4)

# All labels

In [25]:
print(final_df["polarity"].unique())

<ArrowStringArray>
['neutral', 'positive', 'negative', 'conflict']
Length: 4, dtype: str


In [26]:
#Delete rows with "conflict" polarity
final_df = final_df[final_df["polarity"] != "conflict"]

In [27]:
final_df.shape

(5915, 4)

# Preprocess

In [16]:
final_df["polarity"] = final_df["polarity"].str.strip()
final_df["aspect"] = final_df["aspect"].str.replace('"', '')

# Take some for validation

In [30]:
final_df = final_df.head(5000)

In [31]:
final_df.shape

(5000, 4)

# BIO tagging 

In [ ]:
import pandas as pd
import string

# ---------------------------
# TOKENIZER (clean + simple)
# ---------------------------
def tokenize(text):
    return [
        t.strip(string.punctuation).lower()
        for t in text.split()
        if t.strip(string.punctuation)
    ]

# ---------------------------
# FIND ASPECT SPAN
# ---------------------------
def find_span(tokens, aspect_tokens):
    n, m = len(tokens), len(aspect_tokens)

    for i in range(n - m + 1):
        if tokens[i:i+m] == aspect_tokens:
            return i, i + m - 1

    return -1, -1

# ---------------------------
# BIOES TAGGING 
# ---------------------------
def build_tags(tokens, start, end, polarity):
    tags = ["O"] * len(tokens)

    length = end - start + 1

    # SINGLE WORD → S
    if length == 1:
        tags[start] = f"S-{polarity}"

    # MULTI WORD → B I E
    else:
        tags[start] = f"B-{polarity}"

        for i in range(start + 1, end):
            tags[i] = f"I-{polarity}"

        tags[end] = f"E-{polarity}"

    return tags

# ---------------------------
# MAIN CONVERTER
# ---------------------------
def convert(df):
    dataset = []

    for _, row in df.iterrows():
        sentence = row["sentence"]
        aspect = row["aspect"]
        polarity = row["polarity"].upper()  # POSITIVE/NEGATIVE/NEUTRAL

        tokens = tokenize(sentence)
        aspect_tokens = tokenize(aspect)

        start, end = find_span(tokens, aspect_tokens)

        tags = ["O"] * len(tokens)

        if start != -1:
            tags = build_tags(tokens, start, end, polarity)

        dataset.append(list(zip(tokens, tags)))

    return dataset

# Tag nonDATASET sentences to check 

In [41]:
sentence = "The food is absolutely great"
aspect = "food"
polarity = "positive"

tokens = tokenize(sentence)
aspect_tokens = tokenize(aspect)

start, end = find_span(tokens, aspect_tokens)

tags = build_tags(tokens, start, end, polarity.upper())

result = list(zip(tokens, tags))

print(result)

[('the', 'O'), ('food', 'S-POSITIVE'), ('is', 'O'), ('absolutely', 'O'), ('great', 'O')]


In [49]:
sentence = "The service was really slow and the food quality was amazing"
aspect = "food quality"
polarity = "positive"

tokens = tokenize(sentence)
aspect_tokens = tokenize(aspect)

start, end = find_span(tokens, aspect_tokens)
tags = build_tags(tokens, start, end, polarity.upper())

result = list(zip(tokens, tags))

print("TOKENS:", tokens)
print("ASPECT TOKENS:", aspect_tokens)
print("SPAN:", (start, end))
print("RESULT:", result)

TOKENS: ['the', 'service', 'was', 'really', 'slow', 'and', 'the', 'food', 'quality', 'was', 'amazing']
ASPECT TOKENS: ['food', 'quality']
SPAN: (7, 8)
RESULT: [('the', 'O'), ('service', 'O'), ('was', 'O'), ('really', 'O'), ('slow', 'O'), ('and', 'O'), ('the', 'O'), ('food', 'B-POSITIVE'), ('quality', 'E-POSITIVE'), ('was', 'O'), ('amazing', 'O')]


# Tag dataset

In [43]:
bio_data = convert(final_df)

In [50]:
bio_data

[[('i', 'O'),
  ('charge', 'O'),
  ('it', 'O'),
  ('at', 'O'),
  ('night', 'O'),
  ('and', 'O'),
  ('skip', 'O'),
  ('taking', 'O'),
  ('the', 'O'),
  ('cord', 'S-NEUTRAL'),
  ('with', 'O'),
  ('me', 'O'),
  ('because', 'O'),
  ('of', 'O'),
  ('the', 'O'),
  ('good', 'O'),
  ('battery', 'O'),
  ('life', 'O')],
 [('i', 'O'),
  ('charge', 'O'),
  ('it', 'O'),
  ('at', 'O'),
  ('night', 'O'),
  ('and', 'O'),
  ('skip', 'O'),
  ('taking', 'O'),
  ('the', 'O'),
  ('cord', 'O'),
  ('with', 'O'),
  ('me', 'O'),
  ('because', 'O'),
  ('of', 'O'),
  ('the', 'O'),
  ('good', 'O'),
  ('battery', 'B-POSITIVE'),
  ('life', 'E-POSITIVE')],
 [('the', 'O'),
  ('tech', 'O'),
  ('guy', 'O'),
  ('then', 'O'),
  ('said', 'O'),
  ('the', 'O'),
  ('service', 'B-NEGATIVE'),
  ('center', 'E-NEGATIVE'),
  ('does', 'O'),
  ('not', 'O'),
  ('do', 'O'),
  ('1-to-1', 'O'),
  ('exchange', 'O'),
  ('and', 'O'),
  ('i', 'O'),
  ('have', 'O'),
  ('to', 'O'),
  ('direct', 'O'),
  ('my', 'O'),
  ('concern', 'O'),
  ('to

In [53]:
bio_data[1]

[('i', 'O'),
 ('charge', 'O'),
 ('it', 'O'),
 ('at', 'O'),
 ('night', 'O'),
 ('and', 'O'),
 ('skip', 'O'),
 ('taking', 'O'),
 ('the', 'O'),
 ('cord', 'O'),
 ('with', 'O'),
 ('me', 'O'),
 ('because', 'O'),
 ('of', 'O'),
 ('the', 'O'),
 ('good', 'O'),
 ('battery', 'B-POSITIVE'),
 ('life', 'E-POSITIVE')]

In [51]:
print(type(bio_data))

<class 'list'>


In [52]:
len(bio_data)

5000

# Label vocabulary

In [54]:
labels = sorted({
    tag
    for sentence in bio_data
    for _, tag in sentence
})

label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

print(label2id)

{'B-NEGATIVE': 0, 'B-NEUTRAL': 1, 'B-POSITIVE': 2, 'E-NEGATIVE': 3, 'E-NEUTRAL': 4, 'E-POSITIVE': 5, 'I-NEGATIVE': 6, 'I-NEUTRAL': 7, 'I-POSITIVE': 8, 'O': 9, 'S-NEGATIVE': 10, 'S-NEUTRAL': 11, 'S-POSITIVE': 12}


# Model

In [55]:
from transformers import BertTokenizerFast
import torch

tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

class ABSADataset(torch.utils.data.Dataset):
    def __init__(self, data, label2id, max_len=128):
        self.data = data
        self.label2id = label2id
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens_tags = self.data[idx]

        tokens = [t for t, _ in tokens_tags]
        tags = [l for _, l in tokens_tags]

        encoding = tokenizer(
            tokens,
            is_split_into_words=True,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_offsets_mapping=False,
            return_tensors="pt"
        )

        word_ids = encoding.word_ids()

        label_ids = []
        prev_word_id = None

        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)  # ignore
            elif word_id != prev_word_id:
                label_ids.append(self.label2id[tags[word_id]])
            else:
                # same word (subword)
                label_ids.append(self.label2id[tags[word_id]])
            prev_word_id = word_id

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(label_ids)
        }

# BERT + BiLSTM

In [56]:
import torch
import torch.nn as nn
from transformers import BertModel

class BERT_BiLSTM(nn.Module):
    def __init__(self, num_labels):
        super().__init__()

        self.bert = BertModel.from_pretrained("bert-base-uncased")

        self.lstm = nn.LSTM(
            input_size=768,
            hidden_size=256,
            batch_first=True,
            bidirectional=True
        )

        self.classifier = nn.Linear(256 * 2, num_labels)

    def forward(self, input_ids, attention_mask, labels=None):

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        x = outputs.last_hidden_state  # (B, T, 768)

        x, _ = self.lstm(x)            # (B, T, 512)

        logits = self.classifier(x)    # (B, T, num_labels)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fn(
                logits.view(-1, logits.shape[-1]),
                labels.view(-1)
            )

        return {"loss": loss, "logits": logits}

# Training loop

In [57]:
from torch.utils.data import DataLoader
from torch.optim import AdamW

dataset = ABSADataset(bio_data, label2id)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BERT_BiLSTM(num_labels=len(label2id)).to(device)
optimizer = AdamW(model.parameters(), lr=2e-5)

epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids, attention_mask, labels)

        loss = outputs["loss"]
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1, Loss: 272.3239
Epoch 2, Loss: 176.9602
Epoch 3, Loss: 140.5917


In [58]:
model.eval()

BERT_BiLSTM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affin

In [59]:
def predict(sentence):
    tokens = tokenize(sentence)

    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask)

    logits = outputs["logits"][0]  # (T, num_labels)
    preds = torch.argmax(logits, dim=-1).cpu().numpy()

    word_ids = encoding.word_ids()

    results = []
    prev_word_id = None

    for i, word_id in enumerate(word_ids):
        if word_id is None or word_id == prev_word_id:
            continue

        label = id2label[preds[i]]
        token = tokens[word_id]

        results.append((token, label))

        prev_word_id = word_id

    return results

In [60]:
predict("The food was amazing but service was slow")

[('the', 'O'),
 ('food', 'O'),
 ('was', 'O'),
 ('amazing', 'O'),
 ('but', 'O'),
 ('service', 'O'),
 ('was', 'O'),
 ('slow', 'O')]

In [61]:
from seqeval.metrics import classification_report

y_true = []
y_pred = []

for sample in bio_data:
    tokens = [t for t, _ in sample]
    true = [l for _, l in sample]

    pred = [tag for _, tag in predict(" ".join(tokens))]

    y_true.append(true)
    y_pred.append(pred)

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

    NEGATIVE       0.48      0.46      0.47      1476
     NEUTRAL       0.49      0.06      0.11       910
    POSITIVE       0.56      0.37      0.45      2581

   micro avg       0.52      0.34      0.41      4967
   macro avg       0.51      0.30      0.34      4967
weighted avg       0.52      0.34      0.39      4967

